# HHEM-2.1-Open vs. Current RAGAS Faithfulness Setup

Evaluates whether Vectara's HHEM-2.1-Open classifier is a viable replacement/hybrid for our LiteLLM faithfulness guardrail.

Compares three scoring architectures on the same labeled test cases:

| # | Architecture | LLM calls |
|---|---|---|
| 1 | Standalone HHEM (whole answer vs. context) | 0 |
| 2 | `FaithfulnesswithHHEM` (LLM decomposes claims, HHEM verifies) | 1 |
| 3 | Current prod `Faithfulness` (LLM decomposes, LLM verifies) | 2 |

**Runtime:** local Jupyter kernel via VS Code. GPU is optional — HHEM is a small model and runs fine on CPU for a test set this size, just slower (seconds, not minutes).

Reuses the same judge model as production (`groq/openai/gpt-oss-20b` via Groq) for the LLM-calling scorers, so results are directly comparable to the 0.7 threshold already in use.

## 1. Setup

Installs the libraries we need and confirms whether a GPU is available.

- `transformers` + `torch` — to load HHEM-2.1-Open directly from Hugging Face for the standalone scorer.
- `ragas` — provides both `Faithfulness` (current prod scorer) and `FaithfulnesswithHHEM` (hybrid scorer) so we don't reimplement either.
- `openai` — the client `ragas.llms.llm_factory` wraps to talk to the judge model through Groq's OpenAI-compatible endpoint.

Run this in a Python 3.10+ virtualenv/conda env with the Jupyter kernel selected in VS Code (`Python: Select Interpreter`, then pick that env as the notebook's kernel). GPU is optional here — the check just tells you what's available; CPU is fine for a test set this size.

In [ ]:
%pip install -q transformers torch ragas openai

import torch
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("No GPU detected — proceeding on CPU. HHEM inference will be a bit slower but this is fine for a small test set.")

## 2. Judge API key

The two LLM-calling scorers (`FaithfulnesswithHHEM`'s decomposition step, and the current prod `Faithfulness` scorer) need a Groq API key to reach `groq/openai/gpt-oss-20b`, same as production.

Loaded from a local `.env` file (via `python-dotenv`) rather than typed into the notebook or hardcoded — `.env` is already excluded in `.gitignore`, so it never gets committed even by accident.

**Before running the next cell:** create a file `notebooks/.env` (next to this notebook) containing:
```
GROQ_API_KEY=your-key-here
```

If the file or key is missing, the cell fails loudly with a clear error rather than silently proceeding without credentials.

In [ ]:
%pip install -q python-dotenv

import os
from dotenv import load_dotenv

load_dotenv()  # looks for .env in the current working directory (notebooks/)

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise RuntimeError(
        "GROQ_API_KEY not found. Create notebooks/.env with a line: GROQ_API_KEY=your-key-here"
    )
print("Groq API key loaded.")